# 🚀 ORRET — Fine-tuning Pipeline
## SOE Model Training on Google Colab (FREE TPU)
**Model:** Qwen2.5-7B-Instruct (via Unsloth)
**Dataset:** .loop files from SOE Ruche
**Method:** QLoRA — 4-bit quantization, LoRA adapters

---

## 1. Setup Environment

In [ ]:
# Check GPU/TPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.stdout else 'No NVIDIA GPU')

# Check TPU
try:
    import jax
    print(f'JAX TPU: {jax.device_count()} devices')
except:
    print('No TPU detected')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

## 2. Install Dependencies

In [ ]:
!pip install -q unsloth transformers datasets peft accelerate
!pip install -q bitsandbytes scipy  # 4-bit quantization deps
!pip install -q sentencepiece protobuf
!pip install -q pyarrow fastparquet  # For .loop files

# Verify
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## 3. Load .loop Dataset

.loop files from SOE Ruche are binary columnar format.
We convert them to HuggingFace Dataset format.

In [ ]:
import json
import pyarrow as pa
from pathlib import Path

def load_loop_file(path):
    """Load a .loop file and return rows as list of dicts."""
    # Try Arrow IPC format first
    try:
        with pa.memory_map(str(path), 'r') as source:
            reader = pa.ipc.open_file(source)
            table = reader.read_all()
            return table.to_pydict()
    except Exception as e:
        print(f'  Arrow failed: {e}')
    
    # Fallback: JSONL
    jsonl_path = path.with_suffix('.jsonl')
    if jsonl_path.exists():
        rows = []
        with open(jsonl_path) as f:
            for line in f:
                try:
                    rows.append(json.loads(line))
                except:
                    pass
        return rows
    
    return []

# Load all .loop files from Drive
DRIVE_DATA = Path('/content/drive/MyDrive/soe/datasets/')
LOOP_FILES = list(DRIVE_DATA.glob('*.loop')) + list(DRIVE_DATA.glob('*.jsonl'))
print(f'Found {len(LOOP_FILES)} dataset files')

all_rows = []
for lf in LOOP_FILES:
    print(f'  Loading {lf.name}...', end=' ')
    rows = load_loop_file(lf)
    print(f'{len(rows)} rows')
    all_rows.extend(rows)

print(f'\nTotal: {len(all_rows)} examples')

## 4. Format for Training

Convert raw content to instruction-tuning format:
`[INST] prompt [/INST] response`

In [ ]:
from datasets import Dataset

def format_instruction(row):
    """Format a row as instruction-tuning example."""
    content = row.get('content', '')
    source = row.get('source', 'unknown')
    category = row.get('category', 'general')
    
    # Truncate long content
    content = content[:4000]
    
    # Build instruction
    if source == 'github':
        prompt = f'[INST] Analyze this repository README and explain the key technical concepts:\n\n{content[:2000]} [/INST]'
    elif source == 'arxiv':
        prompt = f'[INST] Summarize this research paper abstract and identify the main contribution:\n\n{content[:2000]} [/INST]'
    elif source == 'youtube':
        prompt = f'[INST] Describe the main topics covered in this video:\n\n{content[:2000]} [/INST]'
    elif source == 'rss':
        prompt = f'[INST] Summarize this article in French:\n\n{content[:2000]} [/INST]'
    else:
        prompt = f'[INST] Explain this content in detail:\n\n{content[:2000]} [/INST]'
    
    response = row.get('title', '') + '\n\n' + content[:1500]
    
    return {
        'instruction': prompt,
        'input': '',
        'output': response[:2000],
        'category': category,
        'source': source,
        'quality': row.get('quality_score', 0.5),
    }

# Format all rows
formatted = [format_instruction(r) for r in all_rows if r.get('content')]

# Filter by quality
min_quality = 0.45
formatted = [f for f in formatted if f['quality'] >= min_quality]
print(f'After quality filter: {len(formatted)} examples')

# Create HuggingFace dataset
ds = Dataset.from_list(formatted)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(ds[\'train\'])} | Test: {len(ds[\'test\'])}')

## 5. Load Model + Tokenizer (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Model config
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'  # Or 'unsloth/QQwen25-7B'
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
BIts = 4
DType = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print('Loading model with QLoRA (4-bit)...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DType,
    load_in_4bit = Bits == 4,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
)

print(f'Model loaded! Parameters: {model.num_parameters() / 1e9:.1f}B')

## 6. Training Config

In [ ]:
from trl import SFTTrainer, TrainingArguments
from transformers import TrainingArguments
from transformers import DataCollatorForSeq2Seq

OUTPUT_DIR = '/content/drive/MyDrive/soe/models/orret-v1'

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = 'adamw_8bit',
    weight_decay = 0.01,
    lr_scheduler_type = 'cosine',
    seed = 42,
    output_dir = OUTPUT_DIR,
    report_to = 'none',
    save_strategy = 'epoch',
    save_total_limit = 2,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds['train'],
    eval_dataset = ds['test'],
    dataset_text_field = 'instruction',
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 4,
    packing = True,  # Efficient sequence packing
    args = training_args,
)

## 7. Train! 🚀

In [ ]:
# Show memory stats before training
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('Starting training...')
trainer.train()
print('✅ Training complete!')

## 8. Save LoRA Adapters + Export GGUF

In [ ]:
# Save LoRA adapters
model.save_pretrained(f'{OUTPUT_DIR}/lora')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/lora')
print(f'✅ LoRA saved to {OUTPUT_DIR}/lora')

# Optional: Merge and export to GGUF
print('Merging LoRA into base model...')
from unsloth.chat_templates import get_chat_template

model.save_pretrained_merged(OUTPUT_DIR, tokenizer, save_method = 'merged_4bit_forced')
print(f'✅ Merged 4-bit model saved to {OUTPUT_DIR}')

# Download to local (run this cell to get download link)
from google.colab import files
import shutil

shutil.make_archive('/content/orret-v1-merged', 'zip', OUTPUT_DIR)
files.download('/content/orret-v1-merged.zip')
print('Download started!')

---

**ORRET Fine-tune Pipeline v1.0**
Built with Unsloth + QLoRA + Google Colab (FREE TPU/GPU)

**Next steps:**
1. Download the merged model
2. Copy to `~/soe/models/gguf/`
3. Run locally with: `ollama run orret`

**For a stronger model:**
- Use Qwen2.5-14B or 32B (needs more VRAM)
- Increase dataset size (more .loop files from Ruche)
- Train longer (5+ epochs)
- Lower learning rate (1e-4)